In [1]:
# Reproducibility and environment information (run this cell first)
import random
import subprocess
import sys

SEED = 266
random.seed(SEED)

try:
    commit = subprocess.check_output(
        ["git", "rev-parse", "HEAD"], text=True, stderr=subprocess.DEVNULL
    ).strip()
except (subprocess.CalledProcessError, FileNotFoundError):
    commit = "unavailable (run inside a committed Git checkout)"

print(f"Git commit: {commit}")
print(f"Python: {sys.version.split()[0]}")

try:
    import numpy as np
    np.random.seed(SEED)
except ImportError:
    print("NumPy: not installed")

try:
    import torch
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)
    print(f"PyTorch: {torch.__version__}")
    print(f"CUDA available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"GPU count: {torch.cuda.device_count()}")
        for index in range(torch.cuda.device_count()):
            print(f"GPU {index}: {torch.cuda.get_device_name(index)}")
    else:
        print("GPU: none detected")
except ImportError:
    print("PyTorch: not installed")
    print("CUDA available: unavailable")
    print("GPU: unavailable")

print(f"SEED={SEED}")

Git commit: 7fa8174a6147388b85b4a4f0fd3e79fa2cc56c7d
Python: 3.9.6
NumPy: not installed
PyTorch: not installed
CUDA available: unavailable
GPU: unavailable
SEED=266


# Part 1 - Multimodal Retrieval-Augmented Generation

This notebook builds a grounded question-answering system over three read-only economics PDFs.

## Current checkpoint

The notebook is complete through **Task 1.1.3b: Image Cropping**. Embeddings, retrieval, and answer generation are intentionally left for later checkpoints.

Safety rule: DOC3 Appendix C (page 7) is retained for auditing, but it must be marked `low` trust and must never be used as RAG evidence.

## 1. Locate the Part 1 input files

The data path is resolved at runtime instead of using a personal Google Drive path. The code checks, in order:

1. `DATA266_PART1_DIR` if you set it,
2. the repository's ignored local `data` folder and the current working directory,
3. common Colab and Kaggle locations.

This lets the same notebook run after a Git clone in Colab or with competition data attached in Kaggle.

In [2]:
import json
import os
from pathlib import Path

REQUIRED_INPUTS = {
    "kb_manifest.json",
    "Document.pdf",
    "Document2_Supplement_A_Outturns_2011-2013.pdf",
    "Document3_Briefing_B_Labour_Prices_Policy.pdf",
}


def contains_required_inputs(folder):
    folder = Path(folder)
    return folder.is_dir() and all((folder / name).is_file() for name in REQUIRED_INPUTS)


def find_repo_root(start):
    start = Path(start).resolve()
    for candidate in (start, *start.parents):
        if (candidate / ".git").exists():
            return candidate
    return start


def locate_part1_inputs():
    configured = os.environ.get("DATA266_PART1_DIR")
    repo_root = find_repo_root(Path.cwd())
    candidates = [
        Path(configured).expanduser() if configured else None,
        repo_root / "data",
        Path.cwd(),
        Path.cwd() / "Part1",
        Path("/content"),
        Path("/content/Part1"),
        Path("/mnt/data"),
    ]
    for candidate in candidates:
        if candidate is not None and contains_required_inputs(candidate):
            return candidate.resolve()

    # Kaggle datasets are nested under /kaggle/input/<dataset-name>/.
    kaggle_root = Path("/kaggle/input")
    if kaggle_root.is_dir():
        for manifest_path in kaggle_root.rglob("kb_manifest.json"):
            if contains_required_inputs(manifest_path.parent):
                return manifest_path.parent.resolve()

    raise FileNotFoundError(
        "Part 1 files were not found. Attach the Kaggle data or set "
        "DATA266_PART1_DIR to the folder containing kb_manifest.json and the three PDFs."
    )


REPO_ROOT = find_repo_root(Path.cwd())
INPUT_DIR = locate_part1_inputs()
print(f"Repository root: {REPO_ROOT}")
print(f"Part 1 input directory: {INPUT_DIR}")

Repository root: /Users/dingyuyao/Documents/Codex/2026-10-17/referenced-chatgpt-conversation-this-is-an/work/data266-lab2-team41
Part 1 input directory: /private/tmp/data266-part1-inputs-20261020


## 2. Read and validate the knowledge-base manifest

The manifest is the source of truth for each document's ID, expected page count, and trust level. We do not infer those values from filenames. The assertions make the notebook stop early if the wrong files or an altered manifest are attached.

In [3]:
with (INPUT_DIR / "kb_manifest.json").open(encoding="utf-8") as file:
    kb_manifest = json.load(file)

documents = {item["doc_id"]: item for item in kb_manifest["documents"]}
assert set(documents) == {"DOC1", "DOC2", "DOC3"}

doc3_overrides = documents["DOC3"].get("section_trust_overrides", [])
assert any(
    7 in override.get("pages", []) and override.get("trust_level") == "low"
    for override in doc3_overrides
), "DOC3 page 7 must be marked low trust in the manifest."

for doc_id, metadata in documents.items():
    print(
        f"{doc_id}: trust={metadata['trust_level']}, "
        f"expected_pages={metadata['pages']}, file={metadata['file']}"
    )

DOC1: trust=high, expected_pages=37, file=Document.pdf
DOC2: trust=high, expected_pages=7, file=Document2_Supplement_A_Outturns_2011-2013.pdf
DOC3: trust=medium, expected_pages=7, file=Document3_Briefing_B_Labour_Prices_Policy.pdf


## 3. Extract text one page at a time

Each output record keeps the metadata that later stages need: `doc_id`, 1-based `page`, `trust_level`, `evidence_allowed`, and extracted `text`. Keeping page boundaries now makes later citations such as `DOC2:p4` possible.

PyMuPDF is installed only if the runtime does not already provide it.

In [4]:
try:
    import fitz  # PyMuPDF
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "PyMuPDF"])
    import fitz

print(f"PyMuPDF: {fitz.VersionBind}")

PyMuPDF: 1.26.5


In [5]:
def trust_for_page(document_metadata, page_number):
    trust_level = document_metadata["trust_level"]
    for override in document_metadata.get("section_trust_overrides", []):
        if page_number in override.get("pages", []):
            trust_level = override["trust_level"]
    return trust_level


def extract_document_pages(input_dir, document_metadata):
    pdf_path = Path(input_dir) / document_metadata["file"]
    records = []

    with fitz.open(pdf_path) as pdf:
        actual_pages = pdf.page_count
        expected_pages = document_metadata["pages"]
        if actual_pages != expected_pages:
            raise ValueError(
                f"{document_metadata['doc_id']} has {actual_pages} pages; "
                f"manifest expects {expected_pages}."
            )

        for page_index, pdf_page in enumerate(pdf):
            page_number = page_index + 1
            trust_level = trust_for_page(document_metadata, page_number)
            records.append(
                {
                    "doc_id": document_metadata["doc_id"],
                    "page": page_number,
                    "trust_level": trust_level,
                    "evidence_allowed": trust_level != "low",
                    "text": pdf_page.get_text("text", sort=True).strip(),
                }
            )

    return records


## 4. Run extraction and verify the safety invariant

The final checks verify all 51 pages, confirm that no page is empty, and prove that DOC3 page 7 is the only low-trust page. A short preview is printed for inspection; Chunking starts in the next checkpoint, not here.

In [6]:
page_records = []
for doc_id in ("DOC1", "DOC2", "DOC3"):
    records = extract_document_pages(INPUT_DIR, documents[doc_id])
    page_records.extend(records)
    character_count = sum(len(record["text"]) for record in records)
    empty_pages = [record["page"] for record in records if not record["text"]]
    print(
        f"{doc_id}: extracted_pages={len(records)}, "
        f"characters={character_count:,}, empty_pages={empty_pages}"
    )

assert len(page_records) == 51
assert all(record["text"] for record in page_records), "Every PDF page should contain extractable text."

blocked_pages = [
    (record["doc_id"], record["page"], record["trust_level"])
    for record in page_records
    if not record["evidence_allowed"]
]
assert blocked_pages == [("DOC3", 7, "low")]

print(f"Total extracted pages: {len(page_records)}")
print(f"Pages blocked from RAG evidence: {blocked_pages}")

for doc_id in ("DOC1", "DOC2", "DOC3"):
    first_page = next(record for record in page_records if record["doc_id"] == doc_id)
    preview = " ".join(first_page["text"].split())[:180]
    print(f"{doc_id} page 1 preview: {preview}...")

DOC1: extracted_pages=37, characters=176,179, empty_pages=[]
DOC2: extracted_pages=7, characters=10,424, empty_pages=[]
DOC3: extracted_pages=7, characters=8,279, empty_pages=[]
Total extracted pages: 51
Pages blocked from RAG evidence: [('DOC3', 7, 'low')]
DOC1 page 1 preview: 1-1 The Crisis Table 1-1 gives you output growth rates for the world economy, for advanced econ- omies and for other countries separately, since 2000. As you can see, from 2000 to ...
DOC2 page 1 preview: Supplement A After the Crisis: Outturns, Revisions and Forecast Errors, 2011-2013 Document ID DOC2 Publisher DATA 266 Course Data Desk Edition March 2014 data vintage, published 15...
DOC3 page 1 preview: Briefing B Labour Markets, Prices and Policy Trade-offs after the Crisis Document ID DOC3 Publisher Pacific Policy Institute (policy research group) Date 30 June 2014 Source classi...


## Task 1.1.1 checkpoint

PDF text extraction is complete. The next cells perform only **Task 1.1.2: Chunking**.

## 5. Chunk each page while preserving citation metadata

We use a simple word-based sliding window for this first implementation:

- **300 words per chunk** is a practical starting point that usually leaves room under a 512-token embedding limit; exact token counts will be checked after we select the embedding tokenizer.
- **60 words of overlap** repeats about 20% of the previous chunk so a sentence near a boundary is less likely to lose context.
- **No chunk crosses a page boundary**, so every future answer can retain an exact `DOC#:p#` citation.

We create `all_chunks` for auditability, then create `rag_chunks` by removing every record where `evidence_allowed` is false. Only `rag_chunks` may be sent to the future vector database.

In [7]:
CHUNK_SIZE_WORDS = 300
CHUNK_OVERLAP_WORDS = 60

assert CHUNK_SIZE_WORDS > 0
assert 0 <= CHUNK_OVERLAP_WORDS < CHUNK_SIZE_WORDS


def chunk_page_record(page_record, chunk_size=CHUNK_SIZE_WORDS, overlap=CHUNK_OVERLAP_WORDS):
    words = page_record["text"].split()
    page_chunks = []
    start_word = 0
    chunk_index = 1

    while start_word < len(words):
        end_word = min(start_word + chunk_size, len(words))
        chunk_words = words[start_word:end_word]
        page_chunks.append(
            {
                "chunk_id": (
                    f"{page_record['doc_id']}_p{page_record['page']:03d}_"
                    f"c{chunk_index:03d}"
                ),
                "doc_id": page_record["doc_id"],
                "page": page_record["page"],
                "chunk_index": chunk_index,
                "start_word": start_word,
                "end_word": end_word,
                "word_count": len(chunk_words),
                "trust_level": page_record["trust_level"],
                "evidence_allowed": page_record["evidence_allowed"],
                "text": " ".join(chunk_words),
            }
        )

        if end_word == len(words):
            break

        start_word = end_word - overlap
        chunk_index += 1

    return page_chunks


## 6. Build and validate the chunk collections

The checks below verify unique chunk IDs, complete word coverage for every page, the intended overlap, and the low-trust exclusion rule. They also print a compact summary and one sample chunk.

In [8]:
from collections import Counter

all_chunks = []
for page_record in page_records:
    all_chunks.extend(chunk_page_record(page_record))

chunk_ids = [chunk["chunk_id"] for chunk in all_chunks]
assert len(chunk_ids) == len(set(chunk_ids)), "Chunk IDs must be unique."

for page_record in page_records:
    page_chunks = [
        chunk
        for chunk in all_chunks
        if chunk["doc_id"] == page_record["doc_id"] and chunk["page"] == page_record["page"]
    ]
    assert page_chunks[0]["start_word"] == 0
    assert page_chunks[-1]["end_word"] == len(page_record["text"].split())
    for previous, current in zip(page_chunks, page_chunks[1:]):
        assert current["start_word"] == previous["end_word"] - CHUNK_OVERLAP_WORDS

blocked_chunks = [chunk for chunk in all_chunks if not chunk["evidence_allowed"]]
rag_chunks = [chunk for chunk in all_chunks if chunk["evidence_allowed"]]

assert blocked_chunks, "DOC3 page 7 should produce auditable but blocked chunks."
assert all(
    chunk["doc_id"] == "DOC3"
    and chunk["page"] == 7
    and chunk["trust_level"] == "low"
    for chunk in blocked_chunks
)
assert not any(
    chunk["doc_id"] == "DOC3" and chunk["page"] == 7
    for chunk in rag_chunks
), "DOC3 page 7 must never enter the RAG evidence collection."

all_counts = Counter(chunk["doc_id"] for chunk in all_chunks)
rag_counts = Counter(chunk["doc_id"] for chunk in rag_chunks)

print(f"Chunk size: {CHUNK_SIZE_WORDS} words; overlap: {CHUNK_OVERLAP_WORDS} words")
print(f"All chunks: {len(all_chunks)}")
print(f"RAG evidence chunks: {len(rag_chunks)}")
print(f"Blocked low-trust chunks: {len(blocked_chunks)}")
for doc_id in ("DOC1", "DOC2", "DOC3"):
    print(f"{doc_id}: all={all_counts[doc_id]}, evidence={rag_counts[doc_id]}")

sample_chunk = rag_chunks[0]
sample_preview = sample_chunk["text"][:300]
print("\nSample chunk metadata:")
print({key: value for key, value in sample_chunk.items() if key != "text"})
print(f"Sample text: {sample_preview}...")

Chunk size: 300 words; overlap: 60 words
All chunks: 115
RAG evidence chunks: 114
Blocked low-trust chunks: 1
DOC1: all=101, evidence=101
DOC2: all=7, evidence=7
DOC3: all=7, evidence=6

Sample chunk metadata:
{'chunk_id': 'DOC1_p001_c001', 'doc_id': 'DOC1', 'page': 1, 'chunk_index': 1, 'start_word': 0, 'end_word': 300, 'word_count': 300, 'trust_level': 'high', 'evidence_allowed': True}
Sample text: 1-1 The Crisis Table 1-1 gives you output growth rates for the world economy, for advanced econ- omies and for other countries separately, since 2000. As you can see, from 2000 to 2007 the world economy had a sustained expansion. Annual average world output growth was 3.2%, with advanced economies (...


## Task 1.1.2 checkpoint

Text chunking is complete. The next cells perform only **Task 1.1.3a: Figure/Table Inventory**. Image cropping starts in the next checkpoint.

## 7. Detect numbered figures and tables

Before cropping images, we first build an inventory of every numbered Figure and Table. This prevents us from silently saving only the easy charts while missing tables, side-panel figures, or the corrected version of a figure.

The detector uses two signals:

1. a line begins with a valid label such as `Figure A-3`, `Figure B-5 (corrected)`, or `Table 1-2`;
2. the label uses the PDF's caption font, which separates a real caption from a normal paragraph that merely mentions a figure.

For each item we retain the exact label, caption, PDF page, bounding boxes, trust level, and evidence permission.

In [9]:
import re

VISUAL_LABEL_RE = re.compile(
    r"^\s*((?P<item_type>Figure|Table)\s+"
    r"(?P<series>[12A-C])-(?P<number>\d+)"
    r"(?P<revision>\s*\(corrected\))?)"
    r"(?=\s|$)\s*(?P<caption>.*)$",
    flags=re.IGNORECASE,
)
CAPTION_FONT_MARKERS = ("bold", "-hv", "-bd", "demi")


def text_block_text(block):
    lines = []
    for line in block.get("lines", []):
        line_text = "".join(span["text"] for span in line.get("spans", [])).strip()
        if line_text:
            lines.append(line_text)
    return " ".join(lines).strip()


def caption_label_match(block):
    if block.get("type") != 0 or not block.get("lines"):
        return None

    first_line = block["lines"][0]
    first_line_text = "".join(
        span["text"] for span in first_line.get("spans", [])
    ).strip()
    match = VISUAL_LABEL_RE.match(first_line_text)
    if match is None:
        return None

    uses_caption_font = any(
        any(marker in span.get("font", "").lower() for marker in CAPTION_FONT_MARKERS)
        for span in first_line.get("spans", [])
    )
    return match if uses_caption_font else None


def clean_visual_caption(caption):
    caption = " ".join(caption.split())
    # Infographic text can be merged with the short caption in PDF reading order.
    if " Output:" in caption and " Population:" in caption:
        caption = caption.split(" Output:", maxsplit=1)[0]
    caption = re.sub(
        r"^(?P<word>[A-Za-z]+)\s+(?P=word)(?=[,\s])",
        r"\g<word>",
        caption,
        flags=re.IGNORECASE,
    )
    return caption.strip()


def nearby_caption_block(blocks, label_index, max_vertical_gap=90):
    label_block = blocks[label_index]
    x0, _, x1, y1 = label_block["bbox"]
    candidates = []

    for candidate_index, candidate in enumerate(blocks):
        if candidate_index == label_index or candidate.get("type") != 0:
            continue

        candidate_x0, candidate_y0, candidate_x1, _ = candidate["bbox"]
        vertical_gap = candidate_y0 - y1
        if vertical_gap < -1 or vertical_gap > max_vertical_gap:
            continue

        candidate_text = text_block_text(candidate)
        if (
            not candidate_text
            or candidate_text.lower().startswith("source:")
            or VISUAL_LABEL_RE.match(candidate_text)
            or sum(character.isalpha() for character in candidate_text) < 4
        ):
            continue

        same_column = abs(candidate_x0 - x0) <= 25
        horizontal_overlap = max(0, min(x1, candidate_x1) - max(x0, candidate_x0))
        priority = (
            0 if same_column else 1,
            vertical_gap,
            -horizontal_overlap,
            len(candidate_text),
        )
        candidates.append((priority, candidate))

    if not candidates:
        return None
    return min(candidates, key=lambda item: item[0])[1]


def inventory_document_visuals(input_dir, document_metadata):
    pdf_path = Path(input_dir) / document_metadata["file"]
    inventory = []

    with fitz.open(pdf_path) as pdf:
        for page_index, pdf_page in enumerate(pdf):
            page_number = page_index + 1
            page_dict = pdf_page.get_text("dict", sort=True)
            blocks = [block for block in page_dict["blocks"] if block.get("type") == 0]

            for block_index, block in enumerate(blocks):
                if caption_label_match(block) is None:
                    continue

                full_match = VISUAL_LABEL_RE.match(text_block_text(block))
                label = full_match.group(1)
                caption = full_match.group("caption").strip()
                caption_block = block

                if not caption:
                    caption_block = nearby_caption_block(blocks, block_index)
                    if caption_block is not None:
                        caption = text_block_text(caption_block)

                caption = clean_visual_caption(caption)
                trust_level = trust_for_page(document_metadata, page_number)
                safe_label = re.sub(r"[^A-Za-z0-9]+", "_", label).strip("_").lower()

                inventory.append(
                    {
                        "visual_id": f"{document_metadata['doc_id']}_{safe_label}",
                        "doc_id": document_metadata["doc_id"],
                        "page": page_number,
                        "item_type": full_match.group("item_type").title(),
                        "series": full_match.group("series").upper(),
                        "number": int(full_match.group("number")),
                        "label": label,
                        "caption": caption,
                        "label_bbox": [round(value, 2) for value in block["bbox"]],
                        "caption_bbox": (
                            [round(value, 2) for value in caption_block["bbox"]]
                            if caption_block is not None
                            else None
                        ),
                        "trust_level": trust_level,
                        "evidence_allowed": trust_level != "low",
                    }
                )

    return inventory


## 8. Build and validate the visual inventory

The checks require unique IDs, non-empty captions, and continuous numbering within every document/type/series group. As with text, the complete audit inventory is separate from the evidence-safe inventory.

In [10]:
from collections import defaultdict

visual_inventory = []
for doc_id in ("DOC1", "DOC2", "DOC3"):
    visual_inventory.extend(inventory_document_visuals(INPUT_DIR, documents[doc_id]))

visual_ids = [item["visual_id"] for item in visual_inventory]
assert len(visual_ids) == len(set(visual_ids)), "Visual IDs must be unique."
assert all(item["caption"] for item in visual_inventory), "Every visual needs a caption."

numbered_groups = defaultdict(set)
for item in visual_inventory:
    group_key = (item["doc_id"], item["item_type"], item["series"])
    numbered_groups[group_key].add(item["number"])

for group_key, observed_numbers in numbered_groups.items():
    expected_numbers = set(range(1, max(observed_numbers) + 1))
    assert observed_numbers == expected_numbers, (
        f"Numbering gap for {group_key}: observed {sorted(observed_numbers)}"
    )

blocked_visuals = [item for item in visual_inventory if not item["evidence_allowed"]]
rag_visual_inventory = [item for item in visual_inventory if item["evidence_allowed"]]

assert len(blocked_visuals) == 1
assert (
    blocked_visuals[0]["doc_id"] == "DOC3"
    and blocked_visuals[0]["page"] == 7
    and blocked_visuals[0]["label"] == "Figure C-1"
    and blocked_visuals[0]["trust_level"] == "low"
)
assert not any(item["label"] == "Figure C-1" for item in rag_visual_inventory)

visual_counts = Counter(item["doc_id"] for item in visual_inventory)
rag_visual_counts = Counter(item["doc_id"] for item in rag_visual_inventory)
expected_visual_counts = {"DOC1": 17, "DOC2": 10, "DOC3": 9}
assert dict(visual_counts) == expected_visual_counts, (
    f"Visual inventory count changed: {dict(visual_counts)}"
)

print(f"All numbered Figures/Tables: {len(visual_inventory)}")
print(f"Evidence-safe Figures/Tables: {len(rag_visual_inventory)}")
print(f"Blocked low-trust visuals: {len(blocked_visuals)}")
for doc_id in ("DOC1", "DOC2", "DOC3"):
    print(f"{doc_id}: all={visual_counts[doc_id]}, evidence={rag_visual_counts[doc_id]}")

print("\nDetected inventory:")
for item in visual_inventory:
    permission = "evidence" if item["evidence_allowed"] else "BLOCKED"
    print(
        f"{item['doc_id']}:p{item['page']} | {item['label']} | "
        f"{permission} | {item['caption']}"
    )

All numbered Figures/Tables: 36
Evidence-safe Figures/Tables: 35
Blocked low-trust visuals: 1
DOC1: all=17, evidence=17
DOC2: all=10, evidence=10
DOC3: all=9, evidence=8

Detected inventory:
DOC1:p1 | Table 1-1 | evidence | World Output Growth since 2000
DOC1:p2 | Figure 1-1 | evidence | Stock prices in the United States, the Euro area, and emerging economies, 2007–2010
DOC1:p3 | Figure 1-2 | evidence | Unemployment rates in the United States and the Euro area, 2000–2012
DOC1:p4 | Figure 1-3 | evidence | The United States
DOC1:p5 | Table 1-2 | evidence | Growth, Unemployment, and Inflation in the United States, 1980–2012
DOC1:p5 | Figure 1-4 | evidence | U.S. Federal Budget surpluses as a percent of GDP since 1990
DOC1:p7 | Table 1-3 | evidence | Growth, Unemployment, and Inflation in the Euro Area, 1980–2012
DOC1:p8 | Figure 1-5 | evidence | The Euro area
DOC1:p10 | Figure 1-6 | evidence | China, 2010
DOC1:p11 | Table 1-4 | evidence | Growth and Inflation in China, 1980–2012
DOC1:p20 

## Task 1.1.3a checkpoint

The numbered Figure/Table inventory is complete. The next cells perform only **Task 1.1.3b: Image Cropping**. Embeddings start in a later checkpoint.

## 9. Define crop regions

DOC2 and DOC3 use a consistent report layout: the bold caption is above the visual and a source line is below it. Their crop boundaries can therefore be inferred from the current caption, source line, next visual, and next section heading.

DOC1 uses a textbook layout with side captions, maps, and bottom-of-page tables. For this fixed read-only PDF, we use a small audited crop-coordinate table. These coordinates describe page layout only; they do not contain question answers.

The generated PNG files are runtime artifacts. They are written to Kaggle/Colab working storage, `DATA266_OUTPUT_DIR`, or the ignored local `.runtime` folder—not committed to GitHub.

In [11]:
DOC1_CROP_BBOXES = {
    "Table 1-1": [145, 512, 525, 655],
    "Figure 1-1": [35, 18, 545, 265],
    "Figure 1-2": [20, 18, 525, 235],
    "Figure 1-3": [185, 18, 540, 250],
    "Table 1-2": [145, 25, 520, 170],
    "Figure 1-4": [20, 375, 525, 655],
    "Table 1-3": [145, 532, 520, 655],
    "Figure 1-5": [45, 35, 520, 640],
    "Figure 1-6": [175, 410, 540, 655],
    "Table 1-4": [145, 25, 520, 155],
    "Figure 2-1": [35, 380, 540, 655],
    "Figure 2-2": [20, 370, 525, 655],
    "Figure 2-3": [35, 18, 540, 310],
    "Figure 2-4": [20, 18, 540, 215],
    "Figure 2-5": [20, 18, 450, 245],
    "Figure 2-6": [120, 18, 540, 235],
    "Figure 2-7": [160, 18, 540, 255],
}
SECTION_HEADING_RE = re.compile(r"^[A-C]\.\d+\s")


def choose_runtime_output_root():
    configured = os.environ.get("DATA266_OUTPUT_DIR")
    if configured:
        return Path(configured).expanduser().resolve()
    if Path("/kaggle/working").is_dir():
        return Path("/kaggle/working/data266_lab2_outputs")
    if Path("/content").is_dir():
        return Path("/content/data266_lab2_outputs")
    return REPO_ROOT / ".runtime" / "part1"


RUNTIME_OUTPUT_ROOT = choose_runtime_output_root()
VISUAL_OUTPUT_DIR = RUNTIME_OUTPUT_ROOT / "visuals"
VISUAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Visual output directory: {VISUAL_OUTPUT_DIR}")

Visual output directory: /private/tmp/data266-structure-check-20261020/visuals


In [12]:
def automatic_report_crop_bbox(pdf_page, item, page_items):
    page_rect = pdf_page.rect
    label_y0 = item["label_bbox"][1]
    top = max(0, label_y0 - 8)

    later_items = [
        candidate
        for candidate in page_items
        if candidate["label_bbox"][1] > label_y0
    ]
    next_label_y0 = (
        min(candidate["label_bbox"][1] for candidate in later_items)
        if later_items
        else None
    )

    source_bottoms = []
    section_tops = []
    for block in pdf_page.get_text("blocks", sort=True):
        block_text = " ".join(block[4].split())
        block_top = block[1]
        block_bottom = block[3]
        if block_top <= label_y0:
            continue
        if next_label_y0 is not None and block_top >= next_label_y0:
            continue
        if block_text.lower().startswith("source:"):
            source_bottoms.append(block_bottom + 8)
        if SECTION_HEADING_RE.match(block_text):
            section_tops.append(block_top - 6)

    bottom_candidates = []
    if source_bottoms:
        bottom_candidates.append(min(source_bottoms))
    if next_label_y0 is not None:
        bottom_candidates.append(next_label_y0 - 5)
    if section_tops:
        bottom_candidates.append(min(section_tops))

    bottom = min(bottom_candidates) if bottom_candidates else page_rect.height - 45
    crop_rect = fitz.Rect(70, top, page_rect.width - 70, bottom)
    crop_rect &= page_rect
    if crop_rect.is_empty or crop_rect.height < 40:
        raise ValueError(f"Invalid crop for {item['visual_id']}: {crop_rect}")
    return crop_rect


def crop_bbox_for_item(pdf_page, item, page_items):
    if item["doc_id"] == "DOC1":
        if item["label"] not in DOC1_CROP_BBOXES:
            raise KeyError(f"Missing DOC1 crop coordinates for {item['label']}")
        crop_rect = fitz.Rect(DOC1_CROP_BBOXES[item["label"]])
        crop_rect &= pdf_page.rect
        return crop_rect

    return automatic_report_crop_bbox(pdf_page, item, page_items)


## 10. Render every Figure and Table to PNG

Each crop is rendered at 2× PDF resolution. The resulting asset record keeps the original metadata plus the crop rectangle, runtime image path, and pixel dimensions. `rag_visual_assets` again excludes low-trust content.

In [13]:
RENDER_SCALE = 2.0
visual_assets = []

for doc_id in ("DOC1", "DOC2", "DOC3"):
    document_metadata = documents[doc_id]
    pdf_path = INPUT_DIR / document_metadata["file"]
    document_items = [item for item in visual_inventory if item["doc_id"] == doc_id]

    with fitz.open(pdf_path) as pdf:
        for item in document_items:
            pdf_page = pdf[item["page"] - 1]
            page_items = [
                candidate
                for candidate in document_items
                if candidate["page"] == item["page"]
            ]
            crop_rect = crop_bbox_for_item(pdf_page, item, page_items)
            output_path = VISUAL_OUTPUT_DIR / f"{item['visual_id']}.png"
            pixmap = pdf_page.get_pixmap(
                matrix=fitz.Matrix(RENDER_SCALE, RENDER_SCALE),
                clip=crop_rect,
                alpha=False,
            )
            pixmap.save(output_path)

            asset = dict(item)
            asset.update(
                {
                    "crop_bbox": [round(value, 2) for value in crop_rect],
                    "image_path": str(output_path),
                    "width_px": pixmap.width,
                    "height_px": pixmap.height,
                }
            )
            visual_assets.append(asset)

assert len(visual_assets) == len(visual_inventory) == 36
assert len({asset["image_path"] for asset in visual_assets}) == len(visual_assets)
assert all(Path(asset["image_path"]).is_file() for asset in visual_assets)
assert all(Path(asset["image_path"]).stat().st_size > 0 for asset in visual_assets)
assert all(asset["width_px"] >= 200 and asset["height_px"] >= 100 for asset in visual_assets)

for asset in visual_assets:
    crop_rect = fitz.Rect(asset["crop_bbox"])
    assert crop_rect.contains(fitz.Rect(asset["label_bbox"])), asset["visual_id"]
    assert crop_rect.contains(fitz.Rect(asset["caption_bbox"])), asset["visual_id"]

blocked_visual_assets = [asset for asset in visual_assets if not asset["evidence_allowed"]]
rag_visual_assets = [asset for asset in visual_assets if asset["evidence_allowed"]]

assert len(blocked_visual_assets) == 1
assert blocked_visual_assets[0]["label"] == "Figure C-1"
assert len(rag_visual_assets) == 35
assert not any(asset["label"] == "Figure C-1" for asset in rag_visual_assets)

print(f"Rendered visual assets: {len(visual_assets)}")
print(f"Evidence-safe visual assets: {len(rag_visual_assets)}")
print(f"Blocked low-trust visual assets: {len(blocked_visual_assets)}")
print(
    "Pixel width range: "
    f"{min(asset['width_px'] for asset in visual_assets)}-"
    f"{max(asset['width_px'] for asset in visual_assets)}"
)
print(
    "Pixel height range: "
    f"{min(asset['height_px'] for asset in visual_assets)}-"
    f"{max(asset['height_px'] for asset in visual_assets)}"
)
print(f"Runtime PNG directory: {VISUAL_OUTPUT_DIR}")
print(
    "Blocked asset retained for audit only: "
    f"{blocked_visual_assets[0]['visual_id']}"
)

Rendered visual assets: 36
Evidence-safe visual assets: 35
Blocked low-trust visual assets: 1
Pixel width range: 710-1040
Pixel height range: 241-1210
Runtime PNG directory: /private/tmp/data266-structure-check-20261020/visuals
Blocked asset retained for audit only: DOC3_figure_c_1


## Stop here

Task 1.1.3b is complete after the 36 generated crops pass both the programmatic checks and visual contact-sheet review. The next checkpoint will make the figures searchable; no CLIP/VLM embeddings or vector-database records have been created yet.